# Phase 2 — Post-Training Quantization (PTQ) — Google Colab

**Goal:** baseline both PTQ INT8 and PTQ INT4. PTQ is the cheapest method (no training, just calibrate min/max per tensor and round) — every QAT variant in notebooks 03-05 has to beat it on KL divergence to justify its training cost.

**Inputs (from your Drive):**
- `/content/drive/MyDrive/sqat-baseline/results/baseline/fp32_logits.pt`
- `/content/drive/MyDrive/sqat-baseline/results/baseline/baseline_results.json`

**Outputs (saved at the end to `/content/drive/MyDrive/sqat-ptq/`):**
- `results/ptq_int{4,8}/training_results.json`
- `results/ptq_inference.json`
- `results/ptq_summary.json`

**Estimated time:** ~30 min total (no training; just calibration + perplexity eval per bit-width)."

## 1. Setup

In [ ]:
import os, sys, subprocess
from google.colab import drive

drive.mount('/content/drive')

WORKING_DIR  = "/content"
REPO_DIR     = f"{WORKING_DIR}/scheduled-qat-for-slm"
GITHUB_URL   = "https://github.com/JpCurada/scheduled-qat-for-slm.git"
BASELINE_DIR = "/content/drive/MyDrive/sqat-baseline/results/baseline"
DRIVE_ROOT   = "/content/drive/MyDrive/sqat-ptq"   # this notebook's outputs

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", GITHUB_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

In [ ]:
!pip install -q -e ".[viz]"

In [ ]:
from pathlib import Path
import torch

FP32_LOGITS = Path(BASELINE_DIR) / "fp32_logits.pt"
assert FP32_LOGITS.exists(), (
    f"FP32 logits not found at {FP32_LOGITS}.\n"
    "Run notebook 01 first and ensure it saved outputs to "
    "/content/drive/MyDrive/sqat-baseline/."
)

size_gb = FP32_LOGITS.stat().st_size / 1e9
print(f"FP32 logits OK: {FP32_LOGITS}  ({size_gb:.2f} GB)")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")

## 2. Configuration

PTQ has no training, but we still want the perplexity eval to use a reduced sequence length (512) so KL divergence is computed against logits from notebook 01 at the matching length. We patch the YAML configs in-place and write Kaggle copies under `configs_kaggle/`.

In [ ]:
import yaml

# Reuse the FP32 model cache from sqat-baseline if it's there (saves the ~6 GB
# HuggingFace download). Otherwise the cache lives in the local repo dir and
# downloads on first use.
DRIVE_MODEL_CACHE = "/content/drive/MyDrive/sqat-baseline/models/base"
LOCAL_MODEL_CACHE = f"{REPO_DIR}/models/base"
MODEL_CACHE       = DRIVE_MODEL_CACHE if Path(DRIVE_MODEL_CACHE).exists() else LOCAL_MODEL_CACHE

CHECKPOINT_DIR = f"{REPO_DIR}/models/checkpoints"
RESULTS_DIR    = f"{REPO_DIR}/results"
COLAB_CFG      = Path(REPO_DIR) / "configs_colab" / "ptq"
COLAB_CFG.mkdir(parents=True, exist_ok=True)

PTQ_SEQ_LEN = 512   # match LOGIT_SEQ_LEN from notebook 01

def make_colab_ptq_config(src: str, dst: Path) -> Path:
    with open(src) as f:
        cfg = yaml.safe_load(f)
    cfg["model"]["cache_dir"] = MODEL_CACHE
    cfg["calibration"]["seq_length"] = PTQ_SEQ_LEN
    cfg["calibration"]["num_samples"] = 64    # smaller calib for Colab speed
    cfg["export"]["output_dir"] = f"{REPO_DIR}/models/gguf/"
    with dst.open("w") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    return dst

ptq8_cfg = make_colab_ptq_config("configs/ptq/ptq_int8.yaml", COLAB_CFG / "ptq_int8.yaml")
ptq4_cfg = make_colab_ptq_config("configs/ptq/ptq_int4.yaml", COLAB_CFG / "ptq_int4.yaml")

print("INT8 config:", ptq8_cfg)
print("INT4 config:", ptq4_cfg)
print(f"Model cache: {MODEL_CACHE}")

## 3. PTQ INT8

Calibrates min/max from 64 train sequences, quantizes Linear weights to INT8, and evaluates perplexity + KL divergence vs FP32 logits.

In [ ]:
from src.training.trainer import run_experiment
import torch, gc

results_int8 = run_experiment(
    config_path=str(ptq8_cfg),
    device_str="cuda",
    baseline_logits=str(FP32_LOGITS),
    run_benchmarks=False,
)

print("\nPTQ INT8 results:")
for k, v in results_int8.items():
    print(f"  {k:20s}  {v}")

gc.collect(); torch.cuda.empty_cache()

## 4. PTQ INT4

Same calibration, INT4 weights. Expect a noticeable PPL bump and meaningful KL divergence — that's the gap QAT methods will try to close.

In [ ]:
results_int4 = run_experiment(
    config_path=str(ptq4_cfg),
    device_str="cuda",
    baseline_logits=str(FP32_LOGITS),
    run_benchmarks=False,
)

print("\nPTQ INT4 results:")
for k, v in results_int4.items():
    print(f"  {k:20s}  {v}")

gc.collect(); torch.cuda.empty_cache()

## 5. Comparison table

FP32 baseline numbers come from notebook 01's saved JSON; INT8/INT4 from this notebook.

In [ ]:
import json
import pandas as pd

with open(Path(BASELINE_DIR) / "baseline_results.json") as f:
    fp32 = json.load(f)

rows = [
    {"method": "FP32 baseline", "bits": 32,
     "perplexity": fp32.get("perplexity"), "kl_divergence": 0.0,
     "size_gb": 6.5},
    {"method": "PTQ", "bits": 8,
     "perplexity": results_int8.get("perplexity"),
     "kl_divergence": results_int8.get("kl_divergence"),
     "size_gb": 1.7},
    {"method": "PTQ", "bits": 4,
     "perplexity": results_int4.get("perplexity"),
     "kl_divergence": results_int4.get("kl_divergence"),
     "size_gb": 0.85},
]
df = pd.DataFrame(rows)
df["ppl_delta_pct"] = ((df["perplexity"] / df["perplexity"].iloc[0]) - 1.0) * 100
df.round(4)

In [ ]:
summary_path = Path(RESULTS_DIR) / "ptq_summary.json"
summary_path.parent.mkdir(parents=True, exist_ok=True)
summary = {"int8": results_int8, "int4": results_int4, "fp32": fp32}
with summary_path.open("w") as f:
    json.dump(summary, f, indent=2, default=str)
print(f"Wrote {summary_path}")

## 6. Plotly comparison — PPL, KLD, model size

Same chart will be rendered in every method notebook so the visual comparison stays consistent across notebooks 02–05.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def plot_method_comparison(rows, title, fp32_ppl=None):
    """Standardised three-panel comparison plot used in notebooks 02-05.

    rows: list of dicts with keys: label, perplexity, kl_divergence, size_gb
    """
    labels = [r["label"] for r in rows]
    fig = make_subplots(rows=1, cols=3,
                        subplot_titles=("Perplexity (lower=better)",
                                        "KL Divergence vs FP32 (lower=better)",
                                        "Model size (GB)"))
    fig.add_trace(go.Bar(x=labels, y=[r["perplexity"]    for r in rows],
                         marker_color="#4C72B0", name="PPL",
                         text=[f"{r['perplexity']:.3f}" if r["perplexity"] is not None else "—" for r in rows],
                         textposition="outside"), 1, 1)
    if fp32_ppl is not None:
        fig.add_hline(y=fp32_ppl, line_dash="dash", line_color="black",
                      annotation_text=f"FP32 = {fp32_ppl:.3f}", row=1, col=1)
    fig.add_trace(go.Bar(x=labels, y=[r["kl_divergence"] for r in rows],
                         marker_color="#DD8452", name="KLD",
                         text=[f"{r['kl_divergence']:.4f}" if r["kl_divergence"] is not None else "—" for r in rows],
                         textposition="outside"), 1, 2)
    fig.add_trace(go.Bar(x=labels, y=[r["size_gb"]       for r in rows],
                         marker_color="#55A868", name="Size",
                         text=[f"{r['size_gb']:.2f}" for r in rows],
                         textposition="outside"), 1, 3)
    fig.update_layout(title=title, showlegend=False, height=420,
                      margin=dict(t=80, b=40, l=40, r=20))
    fig.show()

rows = [
    {"label": "FP32 baseline", "perplexity": fp32.get("perplexity"),    "kl_divergence": 0.0,                                 "size_gb": 6.5},
    {"label": "PTQ INT8",      "perplexity": results_int8.get("perplexity"), "kl_divergence": results_int8.get("kl_divergence"), "size_gb": 1.7},
    {"label": "PTQ INT4",      "perplexity": results_int4.get("perplexity"), "kl_divergence": results_int4.get("kl_divergence"), "size_gb": 0.85},
]
plot_method_comparison(rows, "PTQ — quantization quality vs FP32",
                       fp32_ppl=fp32.get("perplexity"))

## 7. Sample inference — see the difference

Run the same five prompts through FP32, PTQ INT8, and PTQ INT4 with greedy decoding (`do_sample=False`). You should see FP32 and INT8 outputs match almost word-for-word, while INT4 starts diverging on harder prompts. This is the qualitative side of the KL-divergence number above.

Models are loaded one at a time and freed before the next loads, so VRAM usage stays under 8 GB throughout. The same `SAMPLE_PROMPTS` list and `generate_with_model` helper appear in notebooks 03–05 — comparing across notebooks is a direct apples-to-apples test.

In [ ]:
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from src.models.model_wrapper import build_model_for_training
from src.utils.config_loader import load_config

SAMPLE_PROMPTS = [
    "The capital of France is",
    "Python is a programming language used for",
    "Once upon a time, in a small village,",
    "The chemical symbol for gold is",
    "To improve your sleep quality, you should",
]
MAX_NEW_TOKENS = 30
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-1.7B", cache_dir=MODEL_CACHE)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def generate_with_model(model, prompts=SAMPLE_PROMPTS, max_new_tokens=MAX_NEW_TOKENS):
    model.eval()
    out = []
    for p in prompts:
        ids = tokenizer(p, return_tensors="pt").input_ids.to(device)
        with torch.no_grad():
            gen = model.generate(
                ids, max_new_tokens=max_new_tokens, do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        text = tokenizer.decode(gen[0][ids.shape[1]:], skip_special_tokens=True).strip()
        out.append(text)
    return out

def free():
    gc.collect(); torch.cuda.empty_cache()

# 1. FP32 baseline (load FP16 to save VRAM; numerically equivalent for greedy decoding)
print("Generating FP32 (FP16 weights for memory) ...")
fp32_model = AutoModelForCausalLM.from_pretrained(
    "HuggingFaceTB/SmolLM2-1.7B", cache_dir=MODEL_CACHE, torch_dtype=torch.float16,
).to(device)
fp32_outs = generate_with_model(fp32_model)
del fp32_model; free()

# 2. PTQ INT8
print("Generating PTQ INT8 ...")
cfg8 = load_config(str(ptq8_cfg)); cfg8.model.cache_dir = MODEL_CACHE
wrap8 = build_model_for_training(cfg8, device)
int8_outs = generate_with_model(wrap8.model)
del wrap8; free()

# 3. PTQ INT4
print("Generating PTQ INT4 ...")
cfg4 = load_config(str(ptq4_cfg)); cfg4.model.cache_dir = MODEL_CACHE
wrap4 = build_model_for_training(cfg4, device)
int4_outs = generate_with_model(wrap4.model)
del wrap4; free()

print("Done.")

In [ ]:
import pandas as pd
from IPython.display import display, HTML

inference_df = pd.DataFrame({
    "Prompt":   SAMPLE_PROMPTS,
    "FP32":     fp32_outs,
    "PTQ INT8": int8_outs,
    "PTQ INT4": int4_outs,
})

# Persist for notebook 07's cross-method inference comparison.
inference_df.to_json(Path(RESULTS_DIR) / "ptq_inference.json", orient="records", indent=2)

display(HTML(inference_df.to_html(index=False, escape=False).replace(
    "<table", '<table style="font-family:monospace; font-size:12px"')))

## Save outputs to Drive

Persists the JSONs and inference table to `/content/drive/MyDrive/sqat-ptq/` so notebook 07 can read them. PTQ doesn't produce a `.pt` checkpoint (the trainer doesn't persist one — see notebook 06 for the GGUF export approach).

Proceed to `03_standard_qat.ipynb` after this cell finishes.

In [ ]:
import shutil

dst = Path(DRIVE_ROOT)
dst.mkdir(parents=True, exist_ok=True)

shutil.copytree(f"{REPO_DIR}/results", dst / "results", dirs_exist_ok=True)
shutil.copytree(COLAB_CFG, dst / "configs_colab" / "ptq", dirs_exist_ok=True)

print(f"Saved to: {dst}")
!ls -lh {dst}/results/ 2>/dev/null
!du -sh {dst}